In [ ]:
# prompt: import tensorflow

import tensorflow as tf
tf.__version__

'2.20.0'

In [ ]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# load the IMDB Datasets

vocab_size = 10000    # use top 10000 word
max_len = 200         # max review lenght

In [ ]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# pad sequences to the same lenght
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

In [ ]:
# build the LSTM Model
model = Sequential([
    Embedding(input_dim = vocab_size, output_dim = 128, input_length=max_len),
    LSTM(128),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# print model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# train the model
history = model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 173s 545ms/step - accuracy: 0.7729 - loss: 0.4728 - val_accuracy: 0.8450 - val_loss: 0.3588
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 210s 570ms/step - accuracy: 0.8915 - loss: 0.2713 - val_accuracy: 0.8742 - val_loss: 0.3060
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 171s 547ms/step - accuracy: 0.9212 - loss: 0.2078 - val_accuracy: 0.8668 - val_loss: 0.3239
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 165s 527ms/step - accuracy: 0.9478 - loss: 0.1459 - val_accuracy: 0.8646 - val_loss: 0.3981
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 164s 524ms/step - accuracy: 0.9619 - loss: 0.1075 - val_accuracy: 0.8418 - val_loss: 0.4622


In [ ]:
# Evaluate the model

loss, accuracy = model.evaluate(x_test, y_test)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

782/782 ━━━━━━━━━━━━━━━━━━━━ 83s 106ms/step - accuracy: 0.8426 - loss: 0.4665
Test Loss: 0.4665
Test Accuracy: 0.8426


In [ ]:
model.save("lstm_model.h5")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("lstm_model.h5")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.datasets import imdb

# Get the original word index from the IMDb dataset
imdb_word_index = imdb.get_word_index()

# Adjust the word index to match how imdb.load_data preprocesses:
# 0 is padding, 1 is start of sequence, 2 is unknown, 3 is unused.
# Actual words start from index 4.
word_to_id = {k: (v + 3) for k, v in imdb_word_index.items()}
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2
word_to_id["<UNUSED>"] = 3

# Create a Tokenizer instance
# `num_words` ensures that only the top `vocab_size-1` words are considered, plus one for OOV.
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")

# Assign the adjusted word_index to the tokenizer
# Filter word_to_id to only include indices within vocab_size, consistent with model training
filtered_word_index = {word: idx for word, idx in word_to_id.items() if idx < vocab_size}
tokenizer.word_index = filtered_word_index

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
new_text = ["This movie is amazing", "Worst experience ever"]

from tensorflow.keras.preprocessing.sequence import pad_sequences

sequences = tokenizer.texts_to_sequences(new_text)
padded = pad_sequences(sequences, maxlen=100)  # same max_len as training

In [ ]:
predictions = model.predict(padded)

for text, pred in zip(new_text, predictions):
    print(text, "->", "Positive" if pred > 0.5 else "Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
This movie is amazing -> Positive
Worst experience ever -> Negative
